# Week 2 Seminar: Cleaning, Merging and Missing Values
**Course:** Data Science & Machine Learning 

Last week you loaded one file and looked at it. This week you load several
files, compare them, and clean what you find:

1. **Look first, then choose the types.** Read a file as text, then let
   pandas guess, then decide yourself.
2. **Excel exports and dates.** The same data saved in different years looks
   different.
3. **Duplicates, relisting and merges.** The same listing can appear twice in
   a month, again next month, and many times after a careless merge.
4. **Missing values.** Three ways to handle them, on three git branches.

In [1]:
import numpy as np
import pandas as pd


---
## Part 1: Look First, Then Choose the Types 

When pandas reads a file it **guesses** the type of every column. The guess
is made from the file it is reading, so two files of the same data can get
different guesses. We look at the file first, as plain text, and then decide.

### 1.1 Read the file as text

`dtype=str` keeps every cell as text. `keep_default_na=False` keeps empty
cells as `""` instead of turning them into missing values. Nothing is
converted, so you see exactly what is in the file.

In [2]:
feb_text = pd.read_csv(
    "./datasets/sale/sale_2026-02.csv.gz",
    encoding="utf-8",
    dtype=str,
    keep_default_na=False
)

In [3]:
feb_text.shape

(39416, 31)

In [4]:
feb_text.head()

,link,date,price,home_type,city,district,furnished,commission,num_rooms,area,...,park,recreation,school,restaurant,supermarket,title_text,post_text,price_m2,source_month,listing_type
0,/d/obyavlenie/srochno-prodaetsya-kvartira-1-3-...,сегодня в 16:48,43000.0,Вторичный рынок,Ташкентская область,Bektemir,True,True,1,42,...,True,True,True,True,True,"Срочно продается квартира,1/3/9,Водник.","Срочно продается квартира на Воднике,1 комнатн...",1023.809523809524,2026-02,sale
1,/d/obyavlenie/srochna-sotiladi-kvartira-ID4dQe...,сегодня в 13:46,35000.0,Вторичный рынок,Ташкентская область,Bektemir,True,True,1,27,...,True,False,True,True,True,Srochna sotiladi kvartira,Srochna sotaman kvartira ozumiki maklermasman ...,1296.296296296296,2026-02,sale
2,/d/obyavlenie/prodaetsya-2-h-komnatnaya-kvarti...,сегодня в 11:03,52000.0,Вторичный рынок,Ташкентская область,Bektemir,True,True,2,60,...,False,False,False,False,False,Продается 2-х комнатная квартира,"Продается в центре Водника, кирпичный дом,77 с...",866.6666666666666,2026-02,sale
3,/d/obyavlenie/prodam-evroremont-kvartir-ID3Uo8...,24-02-2026,50000.0,Вторичный рынок,Ташкентская область,Bektemir,True,True,2,45,...,True,True,True,True,True,Продам евроремонт квартир,2/3/4 массив панелнийда 2 хонали 3 хона килинг...,1111.111111111111,2026-02,sale
4,/d/obyavlenie/prodayu-1-3-9-na-vodnike-orienti...,23-02-2026,38000.0,Вторичный рынок,Ташкентская область,Bektemir,True,True,1,42,...,True,True,True,True,True,"Продаю 1/3/9 на Воднике ориентир книжка, Гупкина","Продаю однокомнатную квартиру 42 квадрата, все...",904.7619047619048,2026-02,sale


Look at a few columns one by one. What kind of values does each hold:
numbers, words, True/False, or a mixture?

In [5]:
feb_text["area"].value_counts()

area
60     1587
50     1446
70     1374
72     1115
58     1025
       ... 
712       1
703       1
224       1
211       1
238       1
Name: count, Length: 326, dtype: int64

In [6]:
feb_text["furnished"].value_counts()

furnished
True     21894
False    17522
Name: count, dtype: int64

In [8]:
feb_text["home_type"].unique()

<StringArray>
[' Вторичный рынок', ' Новостройки']
Length: 2, dtype: str

In [9]:
feb_text["build_year"].value_counts()

build_year
               28654
1980 - 1989     2650
2025            1808
1990 - 2000     1285
1960 - 1979     1013
2023             836
2024             649
2022             499
2020             407
2018             333
2019             290
2021             277
2001 - 2010      265
2017             124
старше 1960      112
2011 - 2014       97
2016              66
2015              51
Name: count, dtype: int64

In [10]:
feb_text["date"].value_counts()

date
24-02-2026         7382
23-02-2026         5472
25-02-2026         4839
22-02-2026         3337
21-02-2026         2795
                   ... 
сегодня в 01:43       1
сегодня в 06:17       1
сегодня в 06:04       1
сегодня в 04:09       1
сегодня в 00:54       1
Name: count, Length: 1118, dtype: int64

Three things to notice:

- `home_type` starts with a space: `" Вторичный рынок"`.
- `build_year` mixes single years (`2025`) with ranges (`1980 - 1989`) and
  one text value (`старше 1960`). It also has many empty cells.
- `date` holds `сегодня в 16:48` ("today at 16:48") next to normal dates.

### 1.2 Let pandas guess

Now the normal way

In [13]:
feb = pd.read_csv("../../Data/sale/sale_2026-02.csv.gz", encoding="utf-8")

In [14]:
feb.dtypes

link                str
date                str
price           float64
home_type           str
city                str
district            str
furnished          bool
commission         bool
num_rooms         int64
area              int64
apart_floor       int64
home_floor        int64
condition           str
build_type          str
build_plan          str
build_year          str
bathroom            str
ceil_height     float64
hospital           bool
playground         bool
kindergarten       bool
park               bool
recreation         bool
school             bool
restaurant         bool
supermarket        bool
title_text          str
post_text           str
price_m2        float64
source_month        str
listing_type        str
dtype: object

`build_year` is text (`str`) because of the ranges. Check that pandas lost
nothing: the number of empty cells in the text version should equal the
number of missing values in the guessed version.

In [15]:
(feb_text["build_year"] == "").sum()

np.int64(28654)

In [16]:
feb["build_year"].isna().sum()

np.int64(28654)

### 1.3 The same guess on three months

One file tells you little about the next. Load January and March the same
way and compare the two columns that matter.

In [17]:
jan = pd.read_csv("../../Data/sale/sale_2026-01.csv.gz", encoding="utf-8")
mar = pd.read_csv("../../Data/sale/sale_2026-03.csv.gz", encoding="utf-8")

In [18]:
jan["area"].dtype

dtype('int64')

In [19]:
feb["area"].dtype

dtype('int64')

In [20]:
mar["area"].dtype

dtype('float64')

In [22]:
jan["build_year"].dtype

<StringDtype(storage='python', na_value=nan)>

In [23]:
feb["build_year"].dtype

<StringDtype(storage='python', na_value=nan)>

In [24]:
mar["build_year"].dtype

dtype('float64')

March is different. `area` is `float64` because it has missing values, and
`int64` cannot hold a missing value:

In [25]:
mar["area"].isna().sum()

np.int64(2542)

And `build_year` is a number, so the ranges are gone. `1960 - 1979` was saved
as `1960`: the left end of the range, now indistinguishable from a flat that
really was built in 1960.

In [26]:
mar["build_year"].value_counts().head(10)

build_year
1980.0    2435
2025.0    1729
1990.0    1288
1960.0     961
2023.0     809
2024.0     613
2022.0     521
2020.0     398
2018.0     328
2021.0     287
Name: count, dtype: int64

### 1.4 Leading spaces

February has the space in front of `home_type`; March does not.

In [27]:
feb["home_type"].unique()

<StringArray>
[' Вторичный рынок', ' Новостройки']
Length: 2, dtype: str

In [28]:
mar["home_type"].unique()

<StringArray>
['Вторичный рынок', 'Новостройки']
Length: 2, dtype: str

Put the two months together and pandas sees **four** kinds of home where there
are two:

In [29]:
both = pd.concat([feb, mar])

In [31]:
both["home_type"].value_counts()

home_type
 Вторичный рынок    19953
 Новостройки        19463
Вторичный рынок     19352
Новостройки         18838
Name: count, dtype: int64

`.str.strip()` removes spaces at both ends.

In [32]:
both["home_type"].str.strip().value_counts()

home_type
Вторичный рынок    39305
Новостройки        38301
Name: count, dtype: int64

**Your turn.** Do the same for `build_type`: count it in `both`, then count
it again after `.str.strip()`.

In [97]:
# TODO: both["build_type"] ...


### 1.5 Choose the types yourself

`dtype=` tells `read_csv` the type of a column, so every file is read the same
way. `"Int64"` (capital I) is a whole-number type that allows missing values.

In [35]:
my_dtypes = {"area": "Int64", "build_year": "str"}

In [36]:
jan2 = pd.read_csv("../../Data/sale/sale_2026-01.csv.gz", encoding="utf-8", dtype=my_dtypes)
feb2 = pd.read_csv("../../Data/sale/sale_2026-02.csv.gz", encoding="utf-8", dtype=my_dtypes)

In [37]:
jan2["area"].dtype

Int64Dtype()

In [38]:
feb2["area"].dtype

Int64Dtype()

**Your turn.** Load March with `my_dtypes` as `mar2`. Check the type of
`area`, and look at `mar2["build_year"].value_counts()`.

In [98]:
from pandas.core.arrays import numpy_
# TODO: mar2 = pd.read_csv(...)


### 1.6 `build_year`: convert after loading

A common first move is `pd.to_numeric(..., errors="coerce")`: it turns
anything that is not a number into a missing value. Count what that costs.

In [49]:
feb["build_year"].isna().sum()

np.int64(28654)

In [50]:
pd.to_numeric(feb["build_year"], errors="coerce").isna().sum()

np.int64(34076)

The difference is the number of ranges (and `старше 1960`) that were
**destroyed**: real information turned into "missing" without a warning.

A better way keeps the first year and records whether the value was a range.
`.str.extract(r"(\d{4})")` takes the first four digits it finds.

In [51]:
feb["build_year_start"] = feb["build_year"].str.extract(r"(\d{4})")[0].astype("Int64")

In [52]:
feb["build_year_is_range"] = feb["build_year"].str.contains(" - ")

In [53]:
feb["build_year_start"].isna().sum()

np.int64(28654)

In [54]:
feb["build_year_is_range"].value_counts()

build_year_is_range
False    34106
True      5310
Name: count, dtype: int64

`build_year_start` has exactly as many missing values as the original column:
nothing was destroyed. The `build_year_is_range` column tells a later reader
which years are exact and which are only the start of a decade.

### 1.7 `category`: when does it save memory?

`astype("category")` stores each distinct value once and a small code for
every row. It helps when a column has few distinct values.

In [55]:
feb["home_type"].nunique()

2

In [56]:
feb["link"].nunique()

39125

In [57]:
feb["home_type"].memory_usage(deep=True)

4792851

In [58]:
feb["home_type"].astype("category").memory_usage(deep=True)

39791

In [59]:
feb["link"].memory_usage(deep=True)

6138052

In [60]:
feb["link"].astype("category").memory_usage(deep=True)

6249362

`home_type` has 2 distinct values in 39,416 rows and shrinks many times.
`link` has almost one distinct value per row and gets **bigger**. The rule is:
compare the number of distinct values with the number of rows before
choosing `category`.

---
## Part 2: Excel Exports and Dates

### 2.1 Three raw Excel exports

`Data/raw/` holds three original exports of the same website, from 2022,
2024 and 2025. Each takes a few seconds to open.

In [61]:
raw22 = pd.read_excel("../../Data/raw/2022-02_tashkent-city_sale.xlsx")
raw24 = pd.read_excel("../../Data/raw/2024-06_tashkent_sale.xlsx")
raw25 = pd.read_excel("../../Data/raw/2025-12_tashkent_sale.xlsx")

In [62]:
raw22.shape

(21292, 31)

In [63]:
raw24.shape

(52042, 29)

In [64]:
raw25.shape

(38881, 29)

An Excel file can hold several sheets. `pd.ExcelFile(...).sheet_names` lists
them. Even the sheet name changed between years.

In [65]:
pd.ExcelFile("../../Data/raw/2022-02_tashkent-city_sale.xlsx").sheet_names

['Sheet1']

In [66]:
pd.ExcelFile("../../Data/raw/2024-06_tashkent_sale.xlsx").sheet_names

['Лист1']

Which columns are in one export and not in the other? `set(...) - set(...)`
answers that.

In [67]:
set(raw22.columns) - set(raw25.columns)

{'day', 'month', 'year'}

In [68]:
set(raw25.columns) - set(raw22.columns)

{'district'}

2022 has no `district` column. Look at what its `city` column holds, and
compare with 2025:

In [69]:
raw22["city"].value_counts()

city
Мирзо-Улугбекский    2908
Юнусабадский         2667
Чиланзарский         2241
Мирабадский          2160
Яшнабадский          2053
Сергелийский         1991
Яккасарайский        1918
Алмазарский          1823
Шайхантахурский      1780
Учтепинский          1460
Бектемирский          291
Name: count, dtype: int64

In [70]:
raw25["city"].value_counts()

city
Ташкентская область                 38835
Ташкент, Сергелийский район             8
Ташкент, Шайхантахурский район          6
Ташкент, Бектемирский район             5
Ташкент, Яккасарайский район            5
Ташкент, Алмазарский район              4
Ташкент, Яшнабадский район              4
Ташкент, Юнусабадский район             4
Ташкент, Учтепинский район              3
Ташкент, Чиланзарский район             2
Ташкент, Мирабадский район              2
Ташкент, Мирзо-Улугбекский район        2
Name: count, dtype: int64

In [71]:
raw25["district"].value_counts()

district
Mirzo-Ulugbek    4928
Yunusobod        4714
Yashnobod        4618
Sergeli          4347
Mirobod          4197
Yakkasaroy       4161
Shayhontohur     3985
Chilonzor        3282
Olmazor          2925
Uchtepa          1012
Bektemir          711
Name: count, dtype: int64

In 2022 the district lived in `city`, in Russian (`Бектемирский`). In 2025 it
lives in `district`, in Latin script (`Bektemir`), and `city` holds the region.
Same website, same idea, different column. You cannot pool these files until
you have written a mapping from one to the other.

**Your turn.** Compare the columns of `raw24` and `raw25`, in both
directions. Then look at the first three values of `link` in `raw22` and in
`raw25`.

In [ ]:
# TODO: set(raw24.columns) - set(raw25.columns), and the other way round
# TODO: raw22["link"].head(3), raw25["link"].head(3)

### 2.2 Dates

Look at the first dates of three months.

In [72]:
jan["date"].head()

0    23-01-2026
1    20-01-2026
2    19-01-2026
3    18-01-2026
4    18-01-2026
Name: date, dtype: str

In [73]:
feb["date"].head()

0    сегодня в 16:48
1    сегодня в 13:46
2    сегодня в 11:03
3         24-02-2026
4         23-02-2026
Name: date, dtype: str

In [74]:
mar["date"].head()

0    2026-03-26
1    2026-03-26
2    2026-03-24
3    2026-03-23
4    2026-03-23
Name: date, dtype: str

January writes day-month-year, March writes year-month-day, and February has
`сегодня в ...` ("today at ...") on some rows. How many?

In [75]:
feb["date"].str.contains("сегодня").sum()

np.int64(4346)

`pd.to_datetime` with an explicit `format=` reads exactly one format and
nothing else. Always give the format; without it pandas guesses, and
`03-02-2026` could become 3 February or 2 March.

In [76]:
jan_dates = pd.to_datetime(jan["date"], format="%d-%m-%Y")

In [77]:
jan_dates.min(), jan_dates.max()

(Timestamp('2025-12-24 00:00:00'), Timestamp('2026-01-27 00:00:00'))

In February, `errors="coerce"` turns the `сегодня в` rows into `NaT` (missing
date). Count them: it should be the same number as above.

In [78]:
feb_dates = pd.to_datetime(feb["date"], format="%d-%m-%Y", errors="coerce")

In [79]:
feb_dates.isna().sum()

np.int64(4346)

"Today" means the day the page was saved, and that day is not in the file. So
those 4,346 dates cannot be recovered exactly: we only know they fall in
February 2026.

In [81]:
feb["date_clean"] = (
    pd.to_datetime(feb["date"], format="%d-%m-%Y", errors="coerce")
    .fillna(pd.Timestamp("2026-02-28"))
)

**Your turn.** Parse March's dates with the right `format=`, and check that
none came back missing.

In [ ]:
# TODO: mar_dates = pd.to_datetime(mar["date"], format=...)

---
## Part 3: Duplicates, Relisting and Merges (20 min)

### 3.1 Duplicates inside one month

`.duplicated()` marks a row that has appeared before. On one column it looks
only at that column; on the whole frame it needs every column to match.

In [82]:
feb["link"].duplicated().sum()

np.int64(291)

In [83]:
feb.duplicated().sum()

np.int64(275)

Most repeated links are exact copies of a row: the page was saved twice. Drop
the exact copies and see what is left.

In [84]:
feb_clean = feb.drop_duplicates()

In [85]:
len(feb), len(feb_clean)

(39416, 39141)

In [86]:
feb_clean["link"].duplicated().sum()

np.int64(16)

Some links are still repeated, with different values. Look at them:

In [87]:
feb_clean[feb_clean["link"].duplicated(keep=False)].sort_values("link").head(6)

,link,date,price,home_type,city,district,furnished,commission,num_rooms,area,...,supermarket,title_text,post_text,price_m2,source_month,listing_type,build_year_start,build_year_is_range,date_is_today,date_clean
27922,/d/obyavlenie/babur-elit-novostroyka-yakkasara...,сегодня в 10:52,180000.0,Новостройки,Ташкентская область,Yakkasaroy,True,False,3,129,...,False,Бабур Элит Новостройка Яккасарай 3х-комн 129квм,Яккасарайский район\nОриентир: Яккасарайский Р...,1395.348837,2026-02,sale,<NA>,False,True,2026-02-28
27911,/d/obyavlenie/babur-elit-novostroyka-yakkasara...,сегодня в 05:52,180000.0,Новостройки,Ташкентская область,Yakkasaroy,True,False,3,129,...,False,Бабур Элит Новостройка Яккасарай 3х-комн 129квм,Яккасарайский район\nОриентир: Яккасарайский Р...,1395.348837,2026-02,sale,<NA>,False,True,2026-02-28
6365,/d/obyavlenie/dupleks-na-mirabade-2komn-1-6-86...,сегодня в 09:18,95500.0,Новостройки,Ташкентская область,Mirobod,True,False,2,86,...,True,"Дуплекс на Мирабаде. 2комн/1/6, 86м2, Евроремо...",Новостройка !!! \nМирабадский район\nпод ключ ...,1110.465116,2026-02,sale,<NA>,False,True,2026-02-28
6375,/d/obyavlenie/dupleks-na-mirabade-2komn-1-6-86...,сегодня в 04:18,95500.0,Новостройки,Ташкентская область,Mirobod,True,False,2,86,...,True,"Дуплекс на Мирабаде. 2комн/1/6, 86м2, Евроремо...",Новостройка !!! \nМирабадский район\nпод ключ ...,1110.465116,2026-02,sale,<NA>,False,True,2026-02-28
11479,/d/obyavlenie/kvartira-it-park-mchs-2-3-7-78-k...,сегодня в 12:41,125000.0,Новостройки,Ташкентская область,Mirzo-Ulugbek,True,False,2,78,...,False,"Квартира IT-Park, МЧС 2/3/7 78 кв.м","Мирзо Улугбекский район\nОриентир-IT-Park, МЧС...",1602.564103,2026-02,sale,<NA>,False,True,2026-02-28
11476,/d/obyavlenie/kvartira-it-park-mchs-2-3-7-78-k...,сегодня в 07:41,125000.0,Новостройки,Ташкентская область,Mirzo-Ulugbek,True,False,2,78,...,False,"Квартира IT-Park, МЧС 2/3/7 78 кв.м","Мирзо Улугбекский район\nОриентир-IT-Park, МЧС...",1602.564103,2026-02,sale,<NA>,False,True,2026-02-28


**Your turn.** Do the same three counts for January: repeated links, exact
copies, and repeated links left after `drop_duplicates()`.

In [ ]:
# TODO: jan["link"].duplicated().sum(), jan.duplicated().sum(), ...

### 3.2 The same listing in the next month

A flat that does not sell stays on the site and appears again next month. Is
any February link also in January?

In [88]:
feb["link"].isin(jan["link"]).sum()

np.int64(0)

Zero, which is not believable. Look at the links:

In [89]:
jan["link"].head(3)

0    /d/obyavlenie/3-hona-bektemir-tumani-ID4gWnr.html
1    /d/obyavlenie/tashkent-bektemirskiy-rayon-4-ko...
2     /d/obyavlenie/prodaetsya-1komnotnye-ID3Qp8h.html
Name: link, dtype: str

In [90]:
feb["link"].head(3)

0    /d/obyavlenie/srochno-prodaetsya-kvartira-1-3-...
1    /d/obyavlenie/srochna-sotiladi-kvartira-ID4dQe...
2    /d/obyavlenie/prodaetsya-2-h-komnatnaya-kvarti...
Name: link, dtype: str

February's links carry an extra `?search_reason=...` part. The part that stays
the same is the listing id: `ID` followed by letters and digits, just before
`.html`. `.str.extract` pulls it out.

In [91]:
jan["listing_id"] = jan["link"].str.extract(r"(ID[0-9A-Za-z]+)\.html")[0]
feb["listing_id"] = feb["link"].str.extract(r"(ID[0-9A-Za-z]+)\.html")[0]

In [92]:
feb["listing_id"].head(3)

0    ID4iAAK
1    ID4dQeu
2    ID4iYNJ
Name: listing_id, dtype: str

In [93]:
feb["listing_id"].isin(jan["listing_id"]).sum()

np.int64(12745)

In [94]:
feb["listing_id"].isin(jan["listing_id"]).mean()

np.float64(0.3233458494012584)

About a third of February's rows were already on the site in January. Use
`listing_id`, not `link`, whenever you follow a listing across months.

**Your turn.** How many of March's rows were already in February?

In [ ]:
# TODO: mar["listing_id"] = ..., then .isin(feb["listing_id"])

### 3.3 Predict the row count of a merge

When you merge two tables on a key, each row on the left is matched with
**every** row on the right that has the same key. So if a key value appears
$a$ times on the left and $b$ times on the right, it gives $a \times b$ rows.
You can compute this **before** merging, from two `value_counts()`.

A small lookup table of district codes, typed by hand. Chilonzor was entered
twice, with two different codes.

In [99]:
codes = pd.DataFrame({
    "district": ["Yunusobod", "Chilonzor", "Chilonzor", "Bektemir"],
    "code": ["YUN", "CHI", "CHL", "BEK"],
})
codes

,district,code
0,Yunusobod,YUN
1,Chilonzor,CHI
2,Chilonzor,CHL
3,Bektemir,BEK


In [100]:
three = feb[feb["district"].isin(["Yunusobod", "Chilonzor", "Bektemir"])]

In [101]:
three["district"].value_counts()

district
Yunusobod    4771
Chilonzor    3455
Bektemir      641
Name: count, dtype: int64

In [102]:
codes["district"].value_counts()

district
Chilonzor    2
Yunusobod    1
Bektemir     1
Name: count, dtype: int64

Predict: Yunusobod rows × 1 + Chilonzor rows × 2 + Bektemir rows × 1.

In [103]:
4771 * 1 + 3455 * 2 + 641 * 1

12322

In [104]:
merged = three.merge(codes, on="district")

In [105]:
len(three), len(merged)

(8867, 12322)

The merge invented 3,455 rows that do not exist, with no warning.
`validate="m:1"` says "each left row matches at most one right row" and
raises an error when that is false:

In [106]:
try:
    three.merge(codes, on="district", validate="m:1")
except Exception as error:
    print(error)

Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  district
Chilonzor ...


Fix the lookup table first (ask which Chilonzor code is correct), then merge.
Put `validate=` on every merge you write.

### 3.4 Splitting a table and joining it back on `link`

The text columns are large, so someone splits February into a table without
text and a table with only the text, and joins them back later on `link`.

In [109]:
no_text = feb.drop(columns=["title_text", "post_text"])
text_only = feb[["link", "title_text", "post_text"]]

In [110]:
len(no_text), len(text_only)

(39416, 39416)

**Your turn.** Before merging, predict the row count: each link appears $a$
times on both sides, so it gives $a \times a$ rows. Use
`feb["link"].value_counts()`. Then merge on `link` and compare.

In [ ]:
# TODO: counts = feb["link"].value_counts(); (counts * counts).sum()
# TODO: rejoined = no_text.merge(text_only, on="link"); len(rejoined)

**Answer in writing.** Why does the rejoined table have more rows than
February? What should you have checked before using `link` as a key, and what
would you do instead of the merge?

---
## Part 4: Missing Values on Three Branches (50 min)

The column is `ceil_height`, ceiling height in metres. Tall ceilings are more
common in new buildings, so it could help a price model. But most rows do not
have it.

### 4.1 How much is missing?

In [111]:
feb["ceil_height"].isna().sum()

np.int64(26407)

In [115]:
feb["ceil_height"].isna().mean()

np.float64(0.6699563628983154)

### 4.2 Is it missing at random?

If the missing values were completely at random (**MCAR**), the share missing
would be about the same in every group. Check two groupings.

In [117]:
feb["ceil_missing"] = feb["ceil_height"].isna()

In [118]:
feb.groupby("district")["ceil_missing"].mean().sort_values()

district
Bektemir         0.383775
Sergeli          0.459540
Uchtepa          0.499513
Olmazor          0.546213
Yashnobod        0.642273
Yunusobod        0.712429
Shayhontohur     0.721611
Mirzo-Ulugbek    0.745082
Mirobod          0.747805
Chilonzor        0.750507
Yakkasaroy       0.767806
Name: ceil_missing, dtype: float64

`pd.qcut(..., 10, labels=False)` puts each row into a price decile, 0 for the
cheapest tenth and 9 for the most expensive.

In [119]:
feb["price_decile"] = pd.qcut(feb["price"], 10, labels=False)

In [120]:
feb.groupby("price_decile")["ceil_missing"].mean()

price_decile
0    0.543195
1    0.614076
2    0.627578
3    0.636624
4    0.669343
5    0.688286
6    0.693607
7    0.726758
8    0.736939
9    0.772168
Name: ceil_missing, dtype: float64

And the mean of the ceiling heights that **are** there, by price decile:

In [121]:
feb.groupby("price_decile")["ceil_height"].mean()

price_decile
0    2.982495
1    2.970344
2    2.974103
3    2.973694
4    3.006733
5    3.021758
6    3.017379
7    3.065347
8    3.140220
9    3.245062
Name: ceil_height, dtype: float64

The share missing changes with district and rises with price, so the values
are **not** missing completely at random. And the heights that are reported
are taller in the top price deciles.

### 4.3 Three strategies, three numbers

For each strategy we record the same three numbers:

| number | code |
|---|---|
| rows used | `len(df)` |
| spread of `ceil_height` | `df["ceil_height"].std()` |
| correlation with log price | `df["ceil_height"].corr(df["log_price"])` |

In [122]:
feb["log_price"] = np.log(feb["price"])

**Strategy 1: delete** the rows where `ceil_height` is missing.

In [123]:
sale_delete = feb.dropna(subset=["ceil_height"])

In [124]:
len(sale_delete)

13009

In [125]:
sale_delete["ceil_height"].std()

np.float64(0.3696376281020177)

In [126]:
sale_delete["ceil_height"].corr(sale_delete["log_price"])

np.float64(0.12341518427604158)

**Strategy 2: fill with the mean** of the reported heights.

In [127]:
sale_mean = feb.copy()
sale_mean["ceil_height"] = sale_mean["ceil_height"].fillna(feb["ceil_height"].mean())

In [128]:
len(sale_mean)

39416

In [129]:
sale_mean["ceil_height"].std()

np.float64(0.21234922165550413)

In [130]:
sale_mean["ceil_height"].corr(sale_mean["log_price"])

np.float64(0.07130326333878587)

**Strategy 3: fill with a group median.** Each missing height gets the median
height of flats with the same district, home type and number of rooms.
`.transform("median")` returns that median on every row. Groups with no
reported height at all get the overall median.

**Your turn.** Write strategy 3 as `sale_cond`, and record the same three
numbers.

In [136]:
# TODO: group_median = feb.groupby(["district", "home_type", "num_rooms"])["ceil_height"].transform("median")
# TODO: sale_cond = feb.copy(); fill with group_median, then with the overall median

### 4.4 Check the lecture's formula

The lecture showed that mean filling shrinks both the spread and the
correlation by $\sqrt{1-m}$, where $m$ is the share missing. Check it.

In [131]:
m = feb["ceil_height"].isna().mean()

In [132]:
sale_delete["ceil_height"].std() * np.sqrt(1 - m)

np.float64(0.21235468993965514)

In [133]:
sale_mean["ceil_height"].std()

np.float64(0.21234922165550413)

**Your turn.** Do the same for the correlation: multiply strategy 1's
correlation by $\sqrt{1-m}$ and compare it with strategy 2's.

In [ ]:
# TODO

### 4.5 Why did strategy 3 go **up**?

Mean filling lowered the correlation, as the formula says. Group-median
filling raised it above the deleted version. Split strategy 3 into the rows
that were really measured and the rows that were filled in.

In [134]:
filled = feb["ceil_height"].isna()

In [138]:
sale_cond[~filled]["ceil_height"].corr(sale_cond[~filled]["log_price"])

np.float64(0.12341518427604158)

In [139]:
sale_cond[filled]["ceil_height"].corr(sale_cond[filled]["log_price"])

np.float64(0.3484434679906787)

The filled rows correlate with price much more strongly than the real ones.
That is not a discovery about flats: the filled value is a median by
district, home type and rooms, and those three columns already predict price.
On the filled rows, "ceiling height" is a small price model wearing a
measurement's name. Mean filling hides signal where you can see it;
group-median filling creates signal where you cannot.

---
## Part 5: Summary

| Command | What it does |
| :--- | :--- |
| `pd.read_csv(path, dtype=str, keep_default_na=False)` | Read every cell as text, convert nothing |
| `pd.read_csv(path, dtype={"col": "Int64"})` | Choose a column's type yourself |
| `df["col"].dtype` | The type of one column |
| `pd.concat([df1, df2])` | Put two tables one under the other |
| `df["col"].str.strip()` | Remove spaces at both ends |
| `pd.to_numeric(s, errors="coerce")` | Convert to numbers; anything else becomes missing |
| `df["col"].str.extract(r"(\d{4})")` | Pull a pattern out of text |
| `pd.read_excel(path)` / `pd.ExcelFile(path).sheet_names` | Load an Excel sheet / list its sheets |
| `set(df1.columns) - set(df2.columns)` | Columns in one table and not the other |
| `pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")` | Read dates in one exact format |
| `df.duplicated()` / `df.drop_duplicates()` | Find / remove repeated rows |
| `s.isin(other)` | Which values also appear in another column |
| `df1.merge(df2, on="key", validate="m:1")` | Merge, and fail if the key is not unique on the right |
| `df.groupby("col")["x"].mean()` | Mean of `x` in each group |
| `pd.qcut(s, 10, labels=False)` | Split into ten equal-sized groups |
| `df.dropna(subset=["col"])` / `s.fillna(value)` | Delete / fill missing values |
